# Part 14, notebook 2 of 6: five ideas, five switches

The post's harness is about 150 lines in the style of the Part 5 loop (`ctx_harness.py`). It is **not** the Deep Agents library; it is the smallest thing that lets me turn each retrieval idea on and off and measure it.

**The picture.** The librarian's rules: may you carry the whole book (payload), or a sticky note saying "shelf 4, third book" (path)? May you read a chapter at a time (paged read)? If a box is too heavy, does it stay in the hallway with a label on the lid (offloading)? May you send an assistant to read one report and come back with two facts (sub-agent)?

This notebook calls the local model server for 3 short live runs at the end. **STAND-IN:** none. **REAL:** archive, tools, model (Qwen3.8-27B FP8 behind vLLM, temperature 0, thinking off).

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (as in the post)
os.environ.setdefault("RUST_LOG", "error"); os.environ.setdefault("LANCE_LOG", "error")   # silence LanceDB deprecation chatter
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
import inspect
import ctx_harness as h
from ctx_eval import STRATEGIES, build_tasks, score
print(inspect.getsource(h.Config))

@dataclass(frozen=True)
class Config:
    name: str
    grep: bool = False
    paged: bool = False
    offload: bool = False
    search_returns_paths: bool = False
    delegate: bool = False
    visual: bool = False     # adds similar_cases (DINOv3 + LanceDB), see ctx_visual.py
    grep_tools: bool = True  # False = search-only arm: the model may not use grep/glob



## The switches turn into different tool lists

`tool_specs(cfg)` builds what the model is *shown*. Same archive, different tools: that is the whole experiment.

In [2]:
for name in ["payload", "harness", "harness+delegate", "search-paths", "v-harness"]:
    cfg = STRATEGIES[name]
    tools = [t["function"]["name"] for t in h.tool_specs(cfg)]
    print(f"{name:20s} {', '.join(tools)}")
print()
print("read_file in `payload`: ", next(t for t in h.tool_specs(STRATEGIES["payload"]) if t["function"]["name"] == "read_file")["function"]["description"])
print("read_file in `harness`: ", next(t for t in h.tool_specs(STRATEGIES["harness"]) if t["function"]["name"] == "read_file")["function"]["description"])

payload              ls, read_file, search_reports
harness              ls, glob, grep, read_file, search_reports
harness+delegate     ls, glob, grep, read_file, search_reports, analyze
search-paths         ls, read_file, search_reports
v-harness            ls, glob, grep, read_file, search_reports, similar_cases

read_file in `payload`:  Read a whole file.
read_file in `harness`:  Read a file. Returns up to 100 lines starting at `offset`; page through with offset.


## The tools, called by hand (no model)

grep finds the row without loading the file, then a paged read shows only the lines around it. Pick the first T1 question's id.

In [3]:
tasks = build_tasks()
t1 = [t for t in tasks if t["task"] == "T1"][0]
tid = t1["q"].split("id ")[1].split("?")[0]
print("question:", t1["q"][:110], "...\ngold:", t1["gold"], "\n")
hit = h.t_grep(tid, "exports/inspection_log.csv", "lines")
print("grep ->", hit)
line_no = int(hit.split(":")[1])
print("\npaged read around it ->")
print(h.t_read_file("exports/inspection_log.csv", max(line_no - 2, 0), 4, paged=True))
whole = h.t_read_file("exports/inspection_log.csv", paged=False)
print(f"\nversus the whole-file read: {len(whole):,} characters, about {h.est_tokens(whole):,} tokens (window is 65,536)")

question: In exports/inspection_log.csv, what are n_voids and void_area for the row with id test_00331? Answer: ANSWER:  ...
gold: {'n_voids': 1, 'void_area': 156} 

grep -> exports/inspection_log.csv:3733:test_00331,test,17393,7,Sneaker,1.68611,0.79869,0.04396,0.92058,0.06576,4.06474,False,1,156,11682,14.09345,0.25651

paged read around it ->
3731: test_00330,test,10296,3,Dress,2.55512,1.55128,0.02607,0.95047,0.0555,1.33284,False,0,0,25480,0.0,0.0
3732: test_00331,test,17393,7,Sneaker,1.68611,0.79869,0.04396,0.92058,0.06576,4.06474,False,1,156,11682,14.09345,0.25651
3733: test_00332,test,62632,8,Bag,2.46385,1.45179,0.04692,0.99134,0.05091,2.57623,False,5,559,35847,15.01211,0.05559
3734: test_00333,test,40969,8,Bag,2.00822,1.4939,0.01675,0.9891,0.03577,4.7824,True,3,189,30009,11.05581,0.04031
[4001 lines in total; showing 3731-3734]

versus the whole-file read: 450,000 characters, about 112,500 tokens (window is 65,536)


[2026-10-26T09:30:32Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning
[2026-10-26T09:30:32Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning
[2026-10-26T09:30:32Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_

## The offload rule, on a real oversized result

Any tool result above 2,000 estimated tokens is written to `large_tool_results/` and the model gets the path plus the first and last five lines. (The article offloads near 20,000 tokens; mine is 2,000 because my window is 65,536. That is a deliberate scaling.) One trap: a paged read of an offloaded file is exempt, otherwise reading it back would offload again, forever.

In [4]:
from types import SimpleNamespace
from ctx_harness import _run_tool
from ctx_corpus import ARCHIVE
call = SimpleNamespace(id="demo_call", function=SimpleNamespace(name="grep", arguments=json.dumps({"pattern": "kept candidate", "path": "reports", "mode": "lines"})))
m = {"tool_calls": 0, "tools_used": {}, "biggest_tool_result_tokens": 0, "offloads": 0}
off = ARCHIVE / "large_tool_results"
res = _run_tool(call, STRATEGIES["harness"], m, off)
print(f"raw result was ~{m['biggest_tool_result_tokens']:,} tokens; offloads: {m['offloads']}\n")
print("the model sees:\n" + res)
import shutil; shutil.rmtree(off, ignore_errors=True)

raw result was ~5,191 tokens; offloads: 1

the model sees:
[result too large (~5191 tokens); saved to large_tool_results/demo_call.txt. Use grep or read_file on it.]
reports/test_00002.md:8:- kept candidate 0: severity critical, confidence 0.7, uncertain false, contrast 25.67, area_px 280
reports/test_00002.md:9:- kept candidate 1: severity major, confidence 0.3, uncertain true, contrast 21.04, area_px 138
reports/test_00005.md:8:- kept candidate 0: severity major, confidence 0.85, uncertain false, contrast 32.45, area_px 135
reports/test_00005.md:9:- kept candidate 1: severity major, confidence 0.9, uncertain false, contrast 35.33, area_px 225
reports/test_00005.md:13:Image test_00005 was flagged for review due to 2 confident defects that survived the inspection rules. The system kept candidates 0 and 1, which are both major severity defects with confidences of 0.85 and 0.9 respectively. Candidate 2 was rejected by the classifier.
...
reports/test_00195.md:8:- kept candidate 0: severi

## Three live runs on the same question

The needle question, three ways. `stuff` pastes the CSV into the prompt, `payload` gives whole-file tools, `harness` gives grep and paged reads. This calls the model server.

In [5]:
import time
def live(strategy):
    cfg = STRATEGIES[strategy]
    t0 = time.time()
    m = h.run(t1["q"], cfg, stuffed=None)
    m["parsed"] = h.parse_answer(m["answer"]); m["correct"] = score(t1, m["parsed"])
    print(f"{strategy:9s} status={m['status']:9s} correct={m['correct']!s:5s} steps={m['steps']} tools={m['tools_used']} "
          f"total prompt tokens={m['total_prompt_tokens']:,} ({time.time()-t0:.1f}s)")
for s in ("payload", "harness"):
    live(s)
# stuffing: the whole CSV in the prompt
from ctx_eval import stuffed_context
m = h.run(t1["q"], STRATEGIES["stuff"], stuffed=stuffed_context("T1"))
print(f"{'stuff':9s} status={m['status']:9s} (the server refused the request: the table is ~113,500 tokens)")

payload   status=overflow  correct=False steps=1 tools={'read_file': 1} total prompt tokens=543 (2.6s)


harness   status=ok        correct=True  steps=3 tools={'grep': 1, 'read_file': 1} total prompt tokens=2,806 (9.2s)


stuff     status=overflow  (the server refused the request: the table is ~113,500 tokens)


**What this shows.** The two designs that hand over whole things (`stuff`, `payload`) cannot even start on a table four times the window. The harness answers in a few steps with a few thousand tokens. That is the article's claim, reproduced on a real table. The next notebook shows where the same checklist *loses*.